In [9]:
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split, StratifiedKFold
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.utils.tensorboard import SummaryWriter

from CustomSpeachDataset import CustomSpeachDataset
from SpeachClassifierModel import SpeachClassifierModel

In [10]:
device = device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [11]:
MODELS_DIR = Path("checkpoints")
MODELS_DIR.mkdir(exist_ok=True)

In [12]:
dataset = CustomSpeachDataset(Path("preprocessed_dataset"), preload=True)
dataset.to(device)

Preloading dataset from disk...


In [13]:
train_indices, test_indices = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.y.cpu(),
    random_state=42
)

class_weights = 1.0 / dataset.label_counts
all_sample_weights = np.array([class_weights[y] for y in dataset.y.cpu()])

y_train = dataset.y.cpu()[train_indices]
train_dataset = torch.utils.data.Subset(dataset, train_indices)
train_sample_weights = all_sample_weights[train_indices]
len(train_dataset), len(y_train)

(16447, 16447)

In [14]:
def train_new_model(writer: SummaryWriter, fold: int, params: dict, epochs: int, train_loader: DataLoader, val_loader: DataLoader, save_dir: Path) -> SpeachClassifierModel:
    model = SpeachClassifierModel(params["dropout_rate"])
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])

    loss_fn = nn.CrossEntropyLoss()

    save_path = save_dir / f"fold{fold}.pth"
    best_val_loss = 99999999999.9

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for data, target in train_loader:
            y_pred = model(data)

            loss = loss_fn(y_pred, target)
            train_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss /= len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for data, target in val_loader:
                y_pred = model(data)
                loss = loss_fn(y_pred, target)
                val_loss += loss.item()
            val_loss /= len(val_loader)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)

        model.best_val_loss = best_val_loss

        writer.add_scalar(f"Loss/train/fold{fold}", train_loss, epoch)
        writer.add_scalar(f"Loss/val/fold{fold}", val_loss, epoch)

    return model

In [15]:
def train_models_5cv(writer: SummaryWriter, params: dict, skf: StratifiedKFold, epochs: int, save_dir: Path) -> float:
    total_val_loss = 0.0

    for fold, (train_index, val_index) in enumerate(skf.split(y_train, y_train)):
        train_sampler = WeightedRandomSampler(
            weights=train_sample_weights[train_index],
            num_samples=len(train_sample_weights[train_index]),
            replacement=True,
        )
        train_loader = DataLoader(train_dataset, batch_size=512, sampler=train_sampler)

        val_sampler = WeightedRandomSampler(
            weights=train_sample_weights[val_index],
            num_samples=len(train_sample_weights[val_index]),
            replacement=True,
        )
        val_loader = DataLoader(train_dataset, batch_size=1024, sampler=val_sampler)

        model = train_new_model(writer, fold, params, epochs, train_loader, val_loader, save_dir)
        total_val_loss += model.best_val_loss

    avl_val_loss = total_val_loss / skf.n_splits
    return avl_val_loss


In [17]:
params = {
    "lr": 1e-4,
    "dropout_rate": 0.3,
}
save_dir = MODELS_DIR / "training"
save_dir.mkdir(parents=True, exist_ok=True)

torch.save(params, save_dir / "model_params.pth")

epochs = 100
skf = StratifiedKFold(n_splits=5, shuffle=True)

with SummaryWriter(f"runs/training/folds") as writer:
    avg_val_loss = train_models_5cv(writer, params, skf, epochs, save_dir)
